# Free Recall Model Fitting

Fit a single model configuration to free recall data, simulate, and generate figures.

In [ ]:
# parameters — injected by papermill
model_name = "Omnibus"
factory_type = "base"  # "base" or "compterm"
bounds = {}  # overridden by papermill with actual bounds dict
data_name = "HealeyKahana2014"
data_query = "data['listtype'] == -1"
data_path = "../data/HealeyKahana2014.h5"
base_params = {
    "start_drift_rate": 1.0,
    "shared_support": 0.0,
    "item_support": 0.0,
    "learning_rate": 0.0,
    "primacy_scale": 0.0,
    "primacy_decay": 0.0,
    "encoding_drift_decrease": 1.0,
    "allow_repeated_recalls": True,
}
redo_fits = False
redo_sims = True
run_tag = "Fitting"
relative_tolerance = 0.001
popsize = 15
num_steps = 1000
cross_rate = 0.9
diff_w = 0.85
best_of = 3
experiment_count = 50
seed = 0
analysis_paths = [
    "jaxcmr.analyses.spc.plot_spc",
    "jaxcmr.analyses.crp.plot_crp",
    "jaxcmr.analyses.pnr.plot_pnr",
]

In [ ]:
import json
import os
import warnings

import jax.numpy as jnp
from jax import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import rcParams

from cru_to_cmr.models.cmr_compare import make_factory as make_base_factory
from cru_to_cmr.models.cmr_compterm import make_factory as make_compterm_factory
from jaxcmr.fitting import ScipyDE as fitting_method_cls
from jaxcmr.loss.sequence_likelihood import (
    MemorySearchLikelihoodLoss as loss_fn_cls,
)
from jaxcmr.summarize import summarize_parameters
from jaxcmr.simulation import simulate_h5_from_h5
from jaxcmr.helpers import find_project_root, import_from_string, load_data, generate_trial_mask

warnings.filterwarnings("ignore")

ROOT = find_project_root()

# Resolve factory and analyses from parameters
model_factory_cls = make_base_factory() if factory_type == "base" else make_compterm_factory()
comparison_analyses = [import_from_string(path) for path in analysis_paths]

## Load data

In [ ]:
fits_dir = f"{ROOT}/fits"
png_dir = f"{ROOT}/figures/png"
tif_dir = f"{ROOT}/figures/tif"
sims_dir = f"{ROOT}/simulations"
for d in [fits_dir, png_dir, tif_dir, sims_dir]:
    os.makedirs(d, exist_ok=True)

data = load_data(data_path)
trial_mask = generate_trial_mask(data, data_query)

max_size = np.max(data["pres_itemnos"])
connections = jnp.zeros((max_size, max_size))

file_model_name = model_name.replace(" ", "_")
fit_path = os.path.join(fits_dir, f"{data_name}_{file_model_name}_{run_tag}.json")
print(f"Model: {model_name}")
print(f"Fit path: {fit_path}")

## Fit model

In [ ]:
if os.path.exists(fit_path) and not redo_fits:
    print("Fit already exists. Loading.")
    with open(fit_path) as f:
        results = json.load(f)
    if "subject" not in results["fits"]:
        results["fits"]["subject"] = results["subject"]
else:
    print("Fitting model...")
    fitter = fitting_method_cls(
        data,
        connections,
        base_params,
        model_factory_cls,
        loss_fn_cls,
        hyperparams={
            "num_steps": num_steps,
            "pop_size": popsize,
            "relative_tolerance": relative_tolerance,
            "cross_over_rate": cross_rate,
            "diff_w": diff_w,
            "progress_bar": True,
            "display_iterations": False,
            "bounds": bounds,
            "best_of": best_of,
        },
    )

    results = fitter.fit(trial_mask)
    results = dict(results)

    results["data_query"] = data_query
    results["model"] = model_name
    results["name"] = f"{data_name}_{file_model_name}_{run_tag}"
    results["relative_tolerance"] = relative_tolerance
    results["popsize"] = popsize
    results["num_steps"] = num_steps
    results["cross_rate"] = cross_rate
    results["diff_w"] = diff_w

    with open(fit_path, "w") as f:
        json.dump(results, f, indent=4)

## Summarize parameters

In [ ]:
query_parameters = list(bounds.keys())
print(summarize_parameters([results], query_parameters, include_std=True, include_ci=True))

## Simulate and plot

In [ ]:
sim_path = os.path.join(sims_dir, f"{data_name}_{model_name}_{run_tag}_seed_{seed}.hdf5")
print(f"Simulation path: {sim_path}")

if os.path.exists(sim_path) and not redo_sims:
    sim = load_data(sim_path)
else:
    rng = random.PRNGKey(seed)
    rng, rng_iter = random.split(rng)
    sim = simulate_h5_from_h5(
        model_factory_cls=model_factory_cls,
        dataset=data,
        features=connections,
        parameters={key: jnp.array(val) for key, val in results["fits"].items()},
        trial_mask=trial_mask,
        experiment_count=experiment_count,
        rng=rng_iter,
    )

In [ ]:
_trial_mask = generate_trial_mask(sim, data_query)

for analysis in comparison_analyses:
    result_name = f"{data_name}_{file_model_name}_{run_tag}"
    analysis_suffix = analysis.__name__[5:]

    # Color version (tif only)
    color_cycle = [each["color"] for each in rcParams["axes.prop_cycle"]]
    axis = analysis(
        datasets=[sim, data],
        trial_masks=[np.array(_trial_mask), np.array(trial_mask)],
        color_cycle=color_cycle,
        labels=["Model", "Data"],
        contrast_name="source",
        axis=None,
    )
    axis.tick_params(labelsize=14)
    axis.set_xlabel(axis.get_xlabel(), fontsize=16)
    axis.set_ylabel(axis.get_ylabel(), fontsize=16)
    tif_path = os.path.join(tif_dir, f"{result_name}_{analysis_suffix}.tif")
    plt.savefig(tif_path, bbox_inches="tight", dpi=300)
    plt.show()

    # Black & white version (png + tif)
    cmap = plt.get_cmap("binary_r")
    bw_colors = [mcolors.to_hex(cmap(x)) for x in np.linspace(0.00, 0.70, 2)]
    axis = analysis(
        datasets=[sim, data],
        trial_masks=[np.array(_trial_mask), np.array(trial_mask)],
        color_cycle=bw_colors,
        labels=["Model", "Data"],
        contrast_name="source",
        axis=None,
    )
    axis.tick_params(labelsize=14)
    axis.set_xlabel(axis.get_xlabel(), fontsize=16)
    axis.set_ylabel(axis.get_ylabel(), fontsize=16)
    bw_png = os.path.join(png_dir, f"bw_{result_name}_{analysis_suffix}.png")
    bw_tif = os.path.join(tif_dir, f"bw_{result_name}_{analysis_suffix}.tif")
    plt.savefig(bw_png, bbox_inches="tight", dpi=300)
    plt.savefig(bw_tif, bbox_inches="tight", dpi=300)
    plt.show()
    plt.close()

    print(f"Saved: {analysis_suffix} (png + tif)")